# 07 | Revisão técnica de versões explícitas

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** inspecionar candidatas e aprovar conscientemente. **Entradas:** campanha04/05; credencial de operador por ambiente. Por padrão apenas inspeciona. **Esperado:** estados distintos VALIDATED/APPROVED, sem homologação acadêmica.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from src.workbench.academic_ui import campaign_result,model_table
result=campaign_result()
display(model_table(result['models']))
APPROVE_DEMO=os.getenv('V4_APPROVE_DEMO','false').lower()=='true'
REASON='Revisão técnica explícita para demonstração POC sintética; sem homologação acadêmica.'


,Curso,Modelo,Versão,Algoritmo,Qualidade,MAE_regular,RMSE_regular,Run,Registry,Manifesto,SHA256
0,2248,hi.demo.v4.c2248.components,3,ridge,VALIDATED,8.646847,10.213234,http://127.0.0.1:5000/#/experiments/4/runs/5ee...,http://127.0.0.1:5000/#/models/hi.demo.v4.c224...,s3://pucpr-poc/lineage/poc/planilha-turmas/v4/...,53d075044bf30874c87a50c856461a312271f391f61517...
1,2348,hi.demo.v4.c2348.components,1,dummy,VALIDATED,15.677778,19.338103,http://127.0.0.1:5000/#/experiments/4/runs/18e...,http://127.0.0.1:5000/#/models/hi.demo.v4.c234...,s3://pucpr-poc/lineage/poc/planilha-turmas/v4/...,df8d37f01859ac4fe1318d92772339249700e05ae0783c...
2,2287,hi.demo.v4.c2287.components,1,ridge,VALIDATED,4.969551,6.196224,http://127.0.0.1:5000/#/experiments/4/runs/b6a...,http://127.0.0.1:5000/#/models/hi.demo.v4.c228...,s3://pucpr-poc/lineage/poc/planilha-turmas/v4/...,4ce75f85c435b53342b04db8815f1c73da53b87e835ef8...


**Executar:** solicitação e revisão autenticadas. Para aprovar, defina `APPROVE_DEMO=True` e reexecute esta célula. **Resultado:** SQL e alias reconciliados; erros deixam o modelo bloqueado.

In [2]:

if APPROVE_DEMO:
    reviews=call('GET','/v1/governance/requests?limit=500')
    for model in result['models']:
        selector={k:model[k] for k in ('model_name','model_version')}
        current=[r for r in reviews if all(str(r[k])==str(v) for k,v in selector.items())]
        if any(r['status']=='APPROVED' for r in current):
            continue
        pending=[r for r in current if r['status'] in ('RECONCILING','REQUESTED') and r['action']=='APPROVE']
        request=pending[0] if pending else call('POST','/v1/governance/requests',dict(**selector,action='APPROVE',reason=REASON,requested_by='jupyter-poc-reviewer'))
        call('POST',f"/v1/governance/requests/{request['approval_id']}/review",dict(approve=True,confirm='REVIEW_EXPLICIT_VERSION'),operator=True)
else:
    display('INSPECT_ONLY: nenhuma aprovação executada.')
display(pd.DataFrame(call('GET','/v1/serving/models')))


,model_key,model_version,course_code,validation_status,run_id,snapshot_id,approval_status,deployment_status,policy,expires_at
0,hi.demo.v4.c2248.components,1,2248,VALIDATED,4f0921ac71de407f98427e1e5fe22f3c,9158d4db-c80b-5638-8aa0-9467c60f6107,None,None,None,None
1,hi.demo.v4.c2248.components,2,2248,VALIDATED,3bdb8b358a3c45e2b0491bafebe134fe,9158d4db-c80b-5638-8aa0-9467c60f6107,None,None,None,None
2,hi.demo.v4.c2248.components,3,2248,VALIDATED,5eee1810f09c40e088b1ed061d517ea1,9158d4db-c80b-5638-8aa0-9467c60f6107,APPROVED,None,None,None
3,hi.demo.v4.c2287.components,1,2287,VALIDATED,b6add4f963cd42579b92da413349ae11,9158d4db-c80b-5638-8aa0-9467c60f6107,APPROVED,None,None,None
4,hi.demo.v4.c2348.components,1,2348,VALIDATED,18e7787eed2043f98756ba71ee58ff7e,9158d4db-c80b-5638-8aa0-9467c60f6107,APPROVED,None,None,None


**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.